In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve().parent
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
%run "./00_setup.ipynb"


In [ ]:
from models.market_model import MarketModel

model = MarketModel()
print("MarketModel features:", model.get_stage2_features())
print("\nThis model outputs ONLY log_error (not p_market_pred) to prevent market-copying.")
print("Rule 11: p_market_pred is dropped before Stage2 input.")


In [ ]:
print("""
## log_error の計算
market_log_error_win = log(p_market_clipped / p_pred_clipped)

where:
  p_market_clipped = clip(p_market, 0.01, 0.99)
  p_pred_clipped = clip(p_pred, 0.01, 0.99)

Properties:
  - 平均 ≈ 0 (p_pred が unbiased なら)
  - 対称分布 (正規近似)
  - 両側クリップで発散防止
""")


In [ ]:
print("""
## p_market_pred を Stage2 に入れない理由

もし p_market_pred を Stage2 の入力に含めると:
  Stage2 は「市場予測をコピーする」だけで高い AUC を達成できる
  → しかし実運用では市場予測は使えない (t-0 で初めて確定)
  → train AUC >> test AUC の過学習が発生

差分 (log_error) のみを使うことで:
  市場の「歪み」だけを学習
  実運用でも使える特征量に限定
""")


In [ ]:
print("""
## 結論: Market Model 差分分析
- MarketModel は log_error のみを出力 (p_market_pred は破棄)
- これにより市場コピー化を防止
- 学習済みモデルの feature importance で log_error の寄与度を確認が必要
""")
